In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [3]:
account = pd.read_csv('../data/account.csv', sep=';')
account.head()

,account_id,district_id,frequency,date
0,576,55,POPLATEK MESICNE,930101
1,3818,74,POPLATEK MESICNE,930101
2,704,55,POPLATEK MESICNE,930101
3,2378,16,POPLATEK MESICNE,930101
4,2632,24,POPLATEK MESICNE,930102


In [4]:
account['date_parsed'] = pd.to_datetime(account['date'], format='%y%m%d')
account.head()

,account_id,district_id,frequency,date,date_parsed
0,576,55,POPLATEK MESICNE,930101,1993-01-01
1,3818,74,POPLATEK MESICNE,930101,1993-01-01
2,704,55,POPLATEK MESICNE,930101,1993-01-01
3,2378,16,POPLATEK MESICNE,930101,1993-01-01
4,2632,24,POPLATEK MESICNE,930102,1993-01-02


# Berka dates are 1993-1999, safely within pandas' default 1969-1999 pivot for 2-digit years.
# Verified: no dates fall outside this range.

In [5]:
account['date_parsed'].dt.year.describe()

count    4500.000000
mean     1995.098222
std         1.483898
min      1993.000000
25%      1993.000000
50%      1996.000000
75%      1996.000000
max      1997.000000
Name: date_parsed, dtype: float64

In [6]:
client = pd.read_csv('../data/client.csv', sep=';')
client.head()

,client_id,birth_number,district_id
0,1,706213,18
1,2,450204,1
2,3,406009,1
3,4,561201,5
4,5,605703,5


In [9]:
client['birth_date'] = pd.to_datetime(
    '19' + client['birth_number'].astype(str).str[:2] + '-' +
    np.where(client['gender']=='F', (client['birth_number'].astype(str).str[2:4].astype(int)-50).astype(str).str.zfill(2), client['birth_number'].astype(str).str[2:4]) + '-' +
    client['birth_number'].astype(str).str[4:6],
    format='%Y-%m-%d'
)
client.head()

,client_id,birth_number,district_id,gender,birth_date
0,1,706213,18,F,1970-12-13
1,2,450204,1,M,1945-02-04
2,3,406009,1,F,1940-10-09
3,4,561201,5,M,1956-12-01
4,5,605703,5,F,1960-07-03


In [10]:
disp = pd.read_csv('../data/disp.csv', sep=';')
card = pd.read_csv('../data/card.csv', sep=';')
loan = pd.read_csv('../data/loan.csv', sep=';')
order = pd.read_csv('../data/order.csv', sep=';')
trans = pd.read_csv('../data/trans.csv', sep=';')
district = pd.read_csv('../data/district.csv', sep=';', header=None)

for name, df in [('disp',disp),('card',card),('loan',loan),('order',order),('trans',trans),('district',district)]:
    print(name, df.shape)

C:\Users\user\AppData\Local\Temp\ipykernel_41180\4177654635.py:5: DtypeWarning: Columns (0: bank) have mixed types. Specify dtype option on import or set low_memory=False.
  trans = pd.read_csv('../data/trans.csv', sep=';')


disp (5369, 4)
card (892, 4)
loan (682, 7)
order (6471, 6)
trans (1056320, 10)
district (78, 16)


In [11]:
district.columns = ['district_id','district_name','region','n_inhabitants',
    'n_muni_lt499','n_muni_500_1999','n_muni_2000_9999','n_muni_gt10000',
    'n_cities','urban_ratio','avg_salary','unemp_95','unemp_96',
    'n_entrepreneurs_per1000','crimes_95','crimes_96']
district.head()

,district_id,district_name,region,n_inhabitants,n_muni_lt499,n_muni_500_1999,n_muni_2000_9999,n_muni_gt10000,n_cities,urban_ratio,avg_salary,unemp_95,unemp_96,n_entrepreneurs_per1000,crimes_95,crimes_96
0,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14,A15,A16
1,1,Hl.m. Praha,Prague,1204953,0,0,0,1,1,100.0,12541,0.29,0.43,167,85677,99107
2,2,Benesov,central Bohemia,88884,80,26,6,2,5,46.7,8507,1.67,1.85,132,2159,2674
3,3,Beroun,central Bohemia,75232,55,26,4,1,5,41.7,8980,1.95,2.21,111,2824,2813
4,4,Kladno,central Bohemia,149893,63,29,6,2,6,67.4,9753,4.64,5.05,109,5244,5892


In [12]:
trans['date_parsed'] = pd.to_datetime(trans['date'], format='%y%m%d')

obs_end = pd.Timestamp('1997-12-31')
label_end = pd.Timestamp('1998-12-31')

trans_obs = trans[trans['date_parsed'] <= obs_end]
trans_label = trans[(trans['date_parsed'] > obs_end) & (trans['date_parsed'] <= label_end)]

active_in_label = trans_label['account_id'].unique()

# only accounts that existed before the observation window ended (avoid brand-new accounts with no history)
account['date_parsed'] = pd.to_datetime(account['date'], format='%y%m%d')
eligible_accounts = account[account['date_parsed'] <= obs_end - pd.Timedelta(days=180)]['account_id']

churn_df = pd.DataFrame({'account_id': eligible_accounts})
churn_df['churned'] = (~churn_df['account_id'].isin(active_in_label)).astype(int)

churn_df['churned'].value_counts(normalize=True)

churned
0    0.998032
1    0.001968
Name: proportion, dtype: float64

In [13]:
tx_count_obs = trans_obs.groupby('account_id').size()
tx_count_label = trans_label.groupby('account_id').size()

churn_df = churn_df.set_index('account_id')
churn_df['tx_count_obs_avg_per_year'] = tx_count_obs.reindex(churn_df.index).fillna(0) / 5  # 1993-97
churn_df['tx_count_label'] = tx_count_label.reindex(churn_df.index).fillna(0)

# churned = label-year activity dropped to <20% of their historical yearly average
churn_df['churned'] = (churn_df['tx_count_label'] < 0.2 * churn_df['tx_count_obs_avg_per_year']).astype(int)
churn_df['churned'].value_counts(normalize=True)

churned
0    0.998032
1    0.001968
Name: proportion, dtype: float64

In [14]:
churn_df[['tx_count_obs_avg_per_year','tx_count_label']].describe()

,tx_count_obs_avg_per_year,tx_count_label
count,4065.000000,4065.000000
mean,35.909717,72.266913
std,22.970725,18.430409
min,1.400000,0.000000
25%,16.600000,61.000000
50%,30.400000,71.000000
75%,54.400000,83.000000
max,111.800000,155.000000


In [15]:
trans['date_parsed'].max()

Timestamp('1998-12-31 00:00:00')

In [16]:
tx_count_1997 = trans_obs[trans_obs['date_parsed'].dt.year == 1997].groupby('account_id').size()
churn_df['tx_count_1997'] = tx_count_1997.reindex(churn_df.index).fillna(0)

churn_df['churned'] = (churn_df['tx_count_label'] < 0.5 * churn_df['tx_count_1997']).astype(int)
churn_df['churned'].value_counts(normalize=True)

churned
0    0.994096
1    0.005904
Name: proportion, dtype: float64

In [18]:
last_tx = trans.groupby('account_id')['trans_date' if 'trans_date' in trans.columns else 'date_parsed'].max()
last_tx.dt.year.value_counts().sort_index()

date_parsed
1996       3
1997       5
1998    4492
Name: count, dtype: int64

In [19]:
bal_label = trans_label.groupby('account_id')['balance'].mean()
churn_df['avg_balance_label'] = bal_label.reindex(churn_df.index)
churn_df['avg_balance_obs'] = trans_obs[trans_obs['date_parsed'].dt.year==1997].groupby('account_id')['balance'].mean().reindex(churn_df.index)
churn_df['churned'] = (churn_df['avg_balance_label'] < 0.3 * churn_df['avg_balance_obs']).astype(int)
churn_df['churned'].value_counts(normalize=True)

churned
0    0.989914
1    0.010086
Name: proportion, dtype: float64

In [20]:
churn_df['balance_ratio'] = churn_df['avg_balance_label'] / churn_df['avg_balance_obs']
churn_df['balance_ratio'].describe()

count    4056.000000
mean        1.099086
std         0.927843
min        -9.493236
25%         0.882300
50%         1.056292
75%         1.224339
max        46.271766
Name: balance_ratio, dtype: float64

In [21]:
# Exclude accounts where 1997 balance was negative/zero — ratio is meaningless there
valid = churn_df[churn_df['avg_balance_obs'] > 0].copy()

# Bottom 10% of balance_ratio = churned (data-driven cutoff, not a guess)
threshold = valid['balance_ratio'].quantile(0.10)
valid['churned'] = (valid['balance_ratio'] <= threshold).astype(int)

print('Threshold (10th percentile):', threshold)
print(valid['churned'].value_counts(normalize=True))

Threshold (10th percentile): 0.7324559238321237
churned
0    0.899926
1    0.100074
Name: proportion, dtype: float64
